# 03 — DistilBERT Fine-Tuning for SMS Spam Detection

## 1. Project Objective

The objective of this notebook is to fine-tune a pretrained DistilBERT transformer model to classify SMS messages as either legitimate (ham) or spam.

This experiment builds on the initial exploratory data analysis and the TF-IDF + Logistic Regression baseline developed earlier in the project.Untitled

## 2. Dataset

- **Source:** UCI SMS Spam Collection
- **Task:** Binary text classification
- **Input:** SMS message text
- **Target:** Message label (`ham` or `spam`)
- **Class mapping:** `ham = 0`, `spam = 1`
- **Dataset size after cleaning:** 5,169 messages

## 3. Experimental Workflow

1. Import required libraries and verify the environment.
2. Load the original dataset and reproduce the established cleaning procedure.
3. Split the dataset into training, validation, and test sets using stratified sampling.
4. Encode the target labels as numeric class IDs.
5. Load the pretrained DistilBERT tokenizer.
6. Tokenize the SMS messages and prepare input tensors.
7. Load the pretrained DistilBERT model with a sequence-classification head.
8. Fine-tune the model using the training set.
9. Monitor validation performance and select the best model checkpoint.
10. Evaluate the selected model on the held-out test set.
11. Compare the results with the TF-IDF + Logistic Regression baseline.
12. Save the trained model and tokenizer for later application integration.

## 4. Evaluation Metrics

The model will be evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

Particular attention will be given to spam recall and spam precision because both missed spam messages and legitimate messages incorrectly flagged as spam matter in a practical spam detector.

## 5. Reproducibility and Data Integrity

- Use a fixed random seed for reproducible data splitting and training where applicable.
- Use stratified sampling to preserve class proportions.
- Keep training, validation, and test data separate.
- Use the validation set for model-development decisions.
- Reserve the test set for final evaluation.
- Keep raw and processed datasets out of version control when excluded by `.gitignore`.
- Record model configuration, training settings, and evaluation results.

## 6. Expected Outcome

A fine-tuned DistilBERT spam classifier, a documented evaluation against the classical machine-learning baseline, and saved model artifacts that can later be integrated into a FastAPI application.


In [1]:
# Python standard library
import random
import json
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Deep learning
import torch
from torch.utils.data import Dataset, DataLoader

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)


In [2]:
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("\nAll packages imported successfully!")

PyTorch: 2.14.1+cpu
NumPy: 2.5.2
Pandas: 3.0.5
CUDA build: None
CUDA available: False

All packages imported successfully!


In [3]:
# Locate  raw dataset
data_path = Path("../data/raw/SMSSpamCollection")

# whether the file exists
print("Dataset exists:", data_path.exists())
print("Dataset path:", data_path.resolve())

Dataset exists: True
Dataset path: /home/student/email-spam-detector/data/raw/SMSSpamCollection


In [4]:
df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

Dataset shape: (5572, 2)

First five rows:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
print("Dataset information:")
df.info()

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64


In [6]:
# Convert text labels into numerical labels
label_mapping = {
    "ham": 0,
    "spam": 1
}

#creates a new column containing those numerical labels
df["label_id"] = df["label"].map(label_mapping)

# Inspect the conversion
print(df[["label", "label_id"]].head())

# Check for labels that were not mapped
print("\nMissing numerical labels:", df["label_id"].isnull().sum())

  label  label_id
0   ham         0
1   ham         0
2  spam         1
3   ham         0
4   ham         0

Missing numerical labels: 0


In [7]:
X = df["message"]
y = df["label_id"]

print("Messages:", X.shape)
print("Labels:", y.shape)

Messages: (5572,)
Labels: (5572,)


In [8]:
# First split: 70% training, 30% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide temporary data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))
print("Total samples:", len(X_train) + len(X_val) + len(X_test))

Training samples: 3900
Validation samples: 836
Testing samples: 836
Total samples: 5572


In [9]:
# Check class proportions in each dataset split
print("Training distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True).sort_index())

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Training distribution:
label_id
0    0.865897
1    0.134103
Name: proportion, dtype: float64

Validation distribution:
label_id
0    0.866029
1    0.133971
Name: proportion, dtype: float64

Testing distribution:
label_id
0    0.866029
1    0.133971
Name: proportion, dtype: float64


In [10]:
# Load the pretrained DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 30522


In [11]:
# Tokenize training messages
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    max_length=128
)

# Tokenize validation messages
val_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    max_length=128
)

# Tokenize test messages
test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    max_length=128
)

In [12]:
print("Training messages:", len(train_encodings["input_ids"]))
print("Validation messages:", len(val_encodings["input_ids"]))
print("Test messages:", len(test_encodings["input_ids"]))

print("\nFirst training message token IDs:")
print(train_encodings["input_ids"][0])

print("\nFirst training message attention mask:")
print(train_encodings["attention_mask"][0])

Training messages: 3900
Validation messages: 836
Test messages: 836

First training message token IDs:
[101, 3125, 999, 9433, 1018, 1006, 2888, 1010, 1021, 1058, 6220, 1016, 2888, 7644, 2007, 1037, 3722, 2915, 2013, 1020, 4210, 2013, 1037, 3413, 2011, 15214, 27052, 2361, 2000, 2507, 9433, 1037, 1016, 3125, 7785, 2044, 6275, 8117, 2015, 1012, 102]

First training message attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [19]:
class SMSDataset(Dataset):

    def __init__(self, encodings, labels):
        # Store the tokenized encodings in this object
        self.encodings = encodings

        # Convert labels to a Python list and store them
        self.labels = labels.tolist()

    def __len__(self):
        # Return the number of labels stored in this object
        return len(self.labels)

    def __getitem__(self, idx):
        # Retrieve the input IDs for the message at index idx
        input_ids = self.encodings["input_ids"][idx]

        # Retrieve the attention mask for the message at index idx
        attention_mask = self.encodings["attention_mask"][idx]

        # Retrieve the label for the message at index idx
        label = self.labels[idx]

        # Convert input IDs into a PyTorch tensor
        input_ids = torch.tensor(input_ids, dtype=torch.long)

        # Convert attention mask into a PyTorch tensor
        attention_mask = torch.tensor(attention_mask, dtype=torch.long)

        # Convert the label into a PyTorch tensor
        label = torch.tensor(label, dtype=torch.long)

        # Return a dictionary containing the tensors
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": label
        }

In [14]:
train_dataset = SMSDataset(train_encodings, y_train)

val_dataset = SMSDataset(val_encodings, y_val)

test_dataset = SMSDataset(test_encodings, y_test)

print(
    "Training length:", len(train_dataset),
    "\nValidation length:", len(val_dataset),
    "\nTesting length:", len(test_dataset)
)

Training length: 3900 
Validation length: 836 
Testing length: 836


In [15]:
sample = train_dataset[0]

print(sample.items())
print(sample.values())
print(sample.keys())

dict_items([('input_ids', tensor([  101,  3125,   999,  9433,  1018,  1006,  2888,  1010,  1021,  1058,
         6220,  1016,  2888,  7644,  2007,  1037,  3722,  2915,  2013,  1020,
         4210,  2013,  1037,  3413,  2011, 15214, 27052,  2361,  2000,  2507,
         9433,  1037,  1016,  3125,  7785,  2044,  6275,  8117,  2015,  1012,
          102])), ('attention_mask', tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])), ('labels', tensor(1))])
dict_values([tensor([  101,  3125,   999,  9433,  1018,  1006,  2888,  1010,  1021,  1058,
         6220,  1016,  2888,  7644,  2007,  1037,  3722,  2915,  2013,  1020,
         4210,  2013,  1037,  3413,  2011, 15214, 27052,  2361,  2000,  2507,
         9433,  1037,  1016,  3125,  7785,  2044,  6275,  8117,  2015,  1012,
          102]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [16]:
print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label shape:", sample["labels"].shape)

Input IDs shape: torch.Size([41])
Attention mask shape: torch.Size([41])
Label shape: torch.Size([])


In [20]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [21]:
data_collator 

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [22]:
BATCH_SIZE = 16

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

In [23]:
batch = next(iter(train_dataloader))

print("\nIndividual values")
print(batch["input_ids"][:2])
print(batch["attention_mask"][:2])
print(batch["labels"][:2])
print("\nShapes")
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)


Individual values
tensor([[  101,  1045,  2572,  2980,  1050,  7109,  2100,  1998,  5627,  1045,
          2444,  2334,  2000,  2017,  1011,  3793,  1037,  7514,  2000,  2963,
          2358,  5339,  2067,  2013,  2033,  5018,  2361,  2566,  5796,  2290,
          5658, 26895, 10288,  5183, 16001, 17299,  2229,  2243,  1024,  6185,
          2692, 27531,  2692,  2581,  2575,  2683,  2581,  2475,  7514,  2644,
          2000,  2203,   102,     0,     0,     0,     0,     0],
        [  101,  1998,  2200, 14780,  1010,  2035,  2057,  6848,  2003,  2090,
          1057,  1998,  1045,  2069,  1012,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [24]:
print("Real tokens in message 1:", batch["attention_mask"][0].sum().item())
print("Real tokens in message 2:", batch["attention_mask"][1].sum().item())

Real tokens in message 1: 53
Real tokens in message 2: 16


In [28]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [29]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [35]:
print("Total parameters:", sum(p.numel() for p in model.parameters()))

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

Total parameters: 66955010
Trainable parameters: 66955010


In [39]:
model.train()

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"]
)

print(outputs.logits)
print(outputs.logits.shape)

tensor([[-0.1494,  0.1440],
        [-0.0640,  0.2043],
        [-0.0662,  0.2590],
        [-0.0294,  0.1244],
        [-0.1174,  0.2323],
        [-0.0954,  0.1297],
        [-0.0856,  0.1784],
        [-0.0767,  0.3732],
        [-0.2177,  0.1754],
        [-0.0707,  0.2077],
        [-0.1227,  0.1892],
        [-0.0509,  0.1126],
        [-0.1484,  0.1447],
        [-0.1386,  0.1417],
        [-0.0707,  0.1211],
        [-0.1395,  0.1992]], grad_fn=<AddmmBackward0>)
torch.Size([16, 2])


In [57]:
print(batch["labels"])

tensor([1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0])


In [41]:
loss_function = torch.nn.CrossEntropyLoss()

In [42]:
loss = loss_function(outputs.logits, batch["labels"])

print("Loss:", loss.item())

Loss: 0.7809534072875977


In [ ]:
### The computational sequence is:
# optimizer.zero_grad()  # Clear previous gradients

# outputs = model(...)   # Forward propagation

# loss = loss_function(...)  # Calculate loss

# loss.backward()        # Calculate gradients

# optimizer.step()       # Update model parameters

In [44]:
print(model.classifier.weight.grad) #Checking the gradient before backpropagation

None


In [45]:
loss.backward()


In [46]:
print(model.classifier.weight.grad.shape)

torch.Size([2, 768])


In [48]:
print(model.classifier.weight.grad)  # Inspecting the actual gradient values

tensor([[-1.5404e-01,  1.7780e-03, -1.0960e-04,  ..., -4.8733e-02,
         -3.8090e-02, -2.1335e-03],
        [ 1.5404e-01, -1.7780e-03,  1.0960e-04,  ...,  4.8733e-02,
          3.8090e-02,  2.1335e-03]])


In [49]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=1e-2
)

In [51]:
optimizer.zero_grad() #clear gradient

In [52]:
print(model.classifier.weight.grad)

None


In [ ]:
EPOCHS = 3   #3 means the model will process all 3,900 training messages three times

In [58]:
def train_one_epoch(model, dataloader, optimizer, loss_function):

    model.train()
    model.zero_grad(set_to_none=True)
    total_loss = 0

    for batch_number, batch in enumerate(dataloader, start=1):

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        loss = loss_function(outputs.logits, batch["labels"])

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch["labels"].size(0)
        
        if batch_number % 20 == 0: # checks the remainder when the batch number is divided by 20
            print(
                f"Processed {batch_number}/{len(dataloader)} batches | "
                f"Batch Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(dataloader.dataset)   #divides the accumulated loss by the number of examples

    return average_loss

In [60]:
training_loss = train_one_epoch(
    model,
    train_dataloader,
    optimizer,
    loss_function
)

print(f"\nAverage Training Loss: {training_loss:.4f}")

KeyboardInterrupt: 